# 6장. 데이터를 보며 질문을 만드는 EDA

이 노트북은 `book/chapters/ch06_eda_questions.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 그래프나 표를 많이 만드는 것이 아니라, 데이터를 보며 **현재 데이터로 답할 수 있는 질문**을 만들고 그 질문을 지표와 코드로 연결하는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없다면 먼저 `python scripts/preprocess_data.py`를 실행하세요.
- EDA 결과표는 `reports/` 폴더에 CSV와 Markdown 보고서로 저장합니다.
- EDA 결과는 최종 결론이 아니라, 다음 분석 질문을 만드는 중간 산출물입니다.


## 1. EDA는 결론보다 질문에 가깝다

EDA는 Exploratory Data Analysis, 즉 탐색적 데이터 분석입니다. 이름 그대로 데이터를 여러 방향에서 탐색하면서 분포, 패턴, 차이, 관계, 이상한 값을 발견하는 과정입니다.

EDA에서 자주 던지는 질문은 다음과 같습니다.

- 데이터는 어떤 변수들로 구성되어 있는가?
- 주요 숫자형 변수의 분포는 어떤가?
- 주요 범주형 변수의 빈도는 어떤가?
- 특정 그룹 간 차이가 있는가?
- 시간에 따른 변화가 있는가?
- 이상하게 큰 값이나 작은 값이 있는가?
- 다음 단계에서 더 깊이 볼 질문은 무엇인가?


## 2. 관찰, 가설, 결론 구분하기

EDA 결과를 해석할 때는 관찰, 가설, 결론을 구분해야 합니다.

| 구분 | 의미 | 예시 |
|---|---|---|
| 관찰 | 데이터에서 직접 확인한 사실 | 전자기기 카테고리의 매출 비중이 가장 높다 |
| 가설 | 관찰을 바탕으로 생각해 볼 가능성 | 전자기기는 단가가 높아 매출 비중이 클 수 있다 |
| 결론 | 추가 검증 후 말할 수 있는 판단 | 단가와 판매 수량을 함께 분석한 결과 매출 차이의 주요 요인은 단가였다 |

이번 장에서는 주로 관찰과 가설을 만들고, 다음 분석 질문으로 연결하는 연습을 합니다.


## 3. 좋은 분석 질문의 조건

좋은 분석 질문은 데이터로 답할 수 있어야 합니다.

| 조건 | 설명 | 예시 |
|---|---|---|
| 데이터 기반 | 현재 데이터로 답할 수 있어야 함 | 월별 매출은 어떻게 변하는가? |
| 구체적 | 분석 대상과 기준이 명확해야 함 | 카테고리별 매출 비중은 어떻게 다른가? |
| 측정 가능 | 지표로 계산할 수 있어야 함 | 주문 수, 총매출, 평균 구매 금액 |
| 해석 가능 | 결과가 다음 판단과 연결되어야 함 | 어떤 상품군을 더 자세히 봐야 하는가? |
| 검증 가능 | 코드로 확인할 수 있어야 함 | `groupby()`로 계산 가능 |


## 4. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'chapter06').is_dir():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')

PROJECT_ROOT = find_project_root(CURRENT_DIR)
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)

현재 실행 위치: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study
프로젝트 루트: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study
전처리 데이터 폴더: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/data/processed
보고서 폴더: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/reports


## 5. 전처리된 데이터 불러오기

6장은 5장에서 저장한 전처리 데이터를 사용합니다. 전처리 파일이 없다면 먼저 아래 명령을 터미널에서 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [2]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')


전처리 데이터 불러오기 완료


## 6. 데이터 크기와 컬럼 확인하기

EDA를 시작하기 전에 각 데이터의 행/열 수와 컬럼명을 다시 확인합니다. CSV로 저장했다가 다시 불러오면 날짜 컬럼이 문자열로 돌아올 수 있으므로 날짜형 변환도 다시 확인합니다.


In [3]:
data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (96, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 7)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[order_items]
shape: (752, 6)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']



In [4]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

if 'signup_date' in customers.columns:
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

if 'order_month' not in orders.columns:
    orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str)

print('order_date 변환 실패:', orders['order_date'].isna().sum())
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())


order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


## 7. 분석 질문을 먼저 표로 정리하기

EDA를 시작하기 전에 어떤 질문을 볼지 먼저 표로 정리하면 분석이 산만해지지 않습니다. 아래 표는 고객, 상품, 주문, 매출, 시간, 고객 가치 관점에서 만들 수 있는 기본 질문입니다.


In [5]:
questions = pd.DataFrame({
    'question': [
        '카테고리별 completed 주문 수량과 금액은 어떻게 다른가?',
        '월별 completed 주문 금액·주문 수·평균 주문 금액은 어떻게 다른가?',
        '결제수단별 completed 주문 수와 평균 주문 금액은 어떻게 다른가?',
    ],
    'analysis_scope': ['completed 주문', 'completed 주문', 'completed 주문'],
    'metric': [
        'total_quantity, total_sales',
        'total_sales, order_count, avg_order_value',
        'order_count, total_sales, avg_order_value',
    ],
    'validation': [
        '카테고리 합계와 source total 비교',
        '월별 합계와 source total 비교',
        '결제수단 합계와 source total 비교',
    ],
})
questions

,question,analysis_scope,metric,validation
0,카테고리별 completed 주문 수량과 금액은 어떻게 다른가?,completed 주문,"total_quantity, total_sales",카테고리 합계와 source total 비교
1,월별 completed 주문 금액·주문 수·평균 주문 금액은 어떻게 다른가?,completed 주문,"total_sales, order_count, avg_order_value",월별 합계와 source total 비교
2,결제수단별 completed 주문 수와 평균 주문 금액은 어떻게 다른가?,completed 주문,"order_count, total_sales, avg_order_value",결제수단 합계와 source total 비교


## 8. 고객 데이터 EDA

먼저 고객 데이터만 따로 살펴봅니다. 하나의 데이터셋 안에서 분포와 빈도를 확인하면 이후 병합 분석을 할 때 기준이 생깁니다.


In [6]:
customers['age'].describe()


count    150.000000
mean      43.913333
std       17.802397
min       18.000000
25%       30.000000
50%       41.000000
75%       56.750000
max      150.000000
Name: age, dtype: float64

In [7]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city


,city,customer_count,customer_ratio
0,부산,23,15.33
1,대구,21,14.00
2,광주,21,14.00
3,서울,18,12.00
4,울산,17,11.33
5,인천,16,10.67
6,수원,10,6.67
7,성남,9,6.00
8,고양,8,5.33
9,대전,5,3.33


In [8]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender


,gender,customer_count,customer_ratio
0,M,80,53.33
1,F,70,46.67


고객 데이터에서 이어질 수 있는 질문은 다음과 같습니다.

- 고객은 어느 도시에 많이 분포하는가?
- 고객의 평균 나이는 어느 정도인가?
- 성별 고객 수는 어떻게 분포하는가?
- 도시별 구매 금액도 차이가 있는가?


## 9. 상품 데이터 EDA

상품 데이터에서는 카테고리와 가격을 먼저 봅니다. 상품 수가 많은 카테고리가 매출도 높은지는 아직 알 수 없습니다. 그것은 주문 상세와 연결한 뒤 확인해야 합니다.


In [9]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category


,category,product_count,product_ratio
0,패션,16,16.67
1,뷰티,16,16.67
2,식품,16,16.67
3,도서,13,13.54
4,전자기기,13,13.54
5,스포츠,10,10.42
6,생활용품,9,9.38
7,전자 기기,1,1.04
8,생활 용품,1,1.04
9,뷰 티,1,1.04


In [10]:
products['price'].describe()


count    9.600000e+01
mean     2.048958e+05
std      1.011848e+06
min      5.000000e+03
25%      5.275000e+04
50%      9.450000e+04
75%      1.532500e+05
max      9.999999e+06
Name: price, dtype: float64

In [11]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price


,category,product_count,avg_price,min_price,max_price
2,뷰티,16,710937.0,18000.0,9999999.0
3,생활 용품,1,124000.0,124000.0,124000.0
6,식품,16,120812.0,11000.0,195000.0
8,전자기기,13,120231.0,38000.0,178000.0
9,패션,16,115125.0,28000.0,197000.0
0,도서,13,103692.0,34000.0,182000.0
5,스포츠,10,78400.0,5000.0,183000.0
4,생활용품,9,69556.0,29000.0,147000.0
1,뷰 티,1,54000.0,54000.0,54000.0
7,전자 기기,1,21000.0,21000.0,21000.0


## 10. 주문 데이터 EDA

주문 데이터에서는 주문 상태, 결제수단, 주문 기간을 확인합니다. 주문 상태 분포는 이후 매출 분석에서 완료 주문만 볼지, 전체 주문을 볼지 결정하는 기준이 됩니다.


In [12]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status


,order_status,order_count,order_ratio
0,completed,180,60.0
1,cancelled,63,21.0
2,refunded,57,19.0


In [13]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method


,payment_method,order_count,order_ratio
0,kakao_pay,90,30.00
1,card,72,24.00
2,bank_transfer,70,23.33
3,naver_pay,68,22.67


In [14]:
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())
print('주문 월 개수:', orders['order_month'].nunique())


주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00
주문 월 개수: 16


## 11. 매출 분석을 위한 데이터 연결

카테고리별 매출을 계산하려면 주문 상세와 상품 정보를 `product_id` 기준으로 연결해야 합니다. 병합 후에는 행 수와 누락 여부를 반드시 확인합니다.


In [15]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_items[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [16]:
order_context = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator='_order_merge',
)

completed_items = order_context.loc[
    order_context['order_status'].eq('completed')
].copy()

sales_items = completed_items.merge(
    products[['product_id', 'product_name', 'category', 'price']],
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator='_product_merge',
)
sales_items['category'] = sales_items['category'].fillna('Unknown')

print('병합 전 order_items:', len(order_items))
print('주문 병합 후:', len(order_context))
print('주문 미매칭:', order_context['_order_merge'].eq('left_only').sum())
print('completed 상세 행:', len(completed_items))
print('completed 고유 주문:', completed_items['order_id'].nunique())
print('completed 주문 기준 금액:', completed_items['line_total'].sum())
print('상품 미매칭:', sales_items['_product_merge'].eq('left_only').sum())

병합 전 order_items: 752
주문 병합 후: 752
주문 미매칭: 2
completed 상세 행: 446
completed 고유 주문: 179
completed 주문 기준 금액: 141340000.0
상품 미매칭: 16


## 12. 카테고리별 매출 EDA

카테고리별 매출은 어떤 상품군이 매출에 많이 기여했는지 보여 줍니다. 다만 매출이 높은 이유가 판매 수량 때문인지, 단가 때문인지는 추가로 확인해야 합니다.


In [17]:
category_sales = (
    sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

category_sales


,category,total_quantity,total_sales,sales_ratio
10,패션,290.0,32940000.0,23.31
7,식품,226.0,25035000.0,17.71
9,전자기기,176.0,21485000.0,15.20
1,도서,174.0,17663000.0,12.50
3,뷰티,168.0,17103000.0,12.10
6,스포츠,165.0,13237000.0,9.37
5,생활용품,108.0,7001000.0,4.95
0,Unknown,34.0,4015000.0,2.84
4,생활 용품,17.0,2108000.0,1.49
2,뷰 티,12.0,648000.0,0.46


In [18]:
category_sales_with_price = category_sales.merge(
    category_price[['category', 'avg_price']],
    on='category',
    how='left',
)

category_sales_with_price


,category,total_quantity,total_sales,sales_ratio,avg_price
0,패션,290.0,32940000.0,23.31,115125.0
1,식품,226.0,25035000.0,17.71,120812.0
2,전자기기,176.0,21485000.0,15.20,120231.0
3,도서,174.0,17663000.0,12.50,103692.0
4,뷰티,168.0,17103000.0,12.10,710937.0
5,스포츠,165.0,13237000.0,9.37,78400.0
6,생활용품,108.0,7001000.0,4.95,69556.0
7,Unknown,34.0,4015000.0,2.84,NaN
8,생활 용품,17.0,2108000.0,1.49,124000.0
9,뷰 티,12.0,648000.0,0.46,54000.0


해석 예시:

> 관찰: 매출 비중이 높은 카테고리가 존재합니다.
> 가설: 해당 카테고리는 판매 수량이 많거나 평균 단가가 높아서 매출이 클 수 있습니다.
> 추가 분석: 카테고리별 판매 수량과 평균 단가를 함께 비교해야 합니다.


## 13. 월별 매출과 주문 수 EDA

월별 매출을 보려면 주문 상세와 주문 정보를 `order_id` 기준으로 연결합니다. 월별 매출은 시간 흐름을 보여 주지만, 원인을 바로 설명하지는 않습니다.


In [19]:
order_sales = completed_items.copy()
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = (
    order_sales['order_date'].dt.to_period('M').astype('string').fillna('Unknown')
)

print('completed 상세 행:', len(order_sales))
print('order_date 누락:', order_sales['order_date'].isna().sum())
print('분석 월:', order_sales['order_month'].nunique())

completed 상세 행: 446
order_date 누락: 6
분석 월: 16


In [20]:
monthly_sales = (
    order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)

monthly_sales


,order_month,total_sales,order_count,avg_order_value
0,2025-07,9561000.0,11,869182.0
1,2025-08,9362000.0,17,550706.0
2,2025-09,5299000.0,4,1324750.0
3,2025-10,12629000.0,17,742882.0
4,2025-11,9583000.0,16,598938.0
5,2025-12,8581000.0,12,715083.0
6,2026-01,8135000.0,9,903889.0
7,2026-02,7689000.0,12,640750.0
8,2026-03,7247000.0,13,557462.0
9,2026-04,11459000.0,14,818500.0


월별 매출이 특정 월에 높아졌다면 바로 원인을 단정하지 말고, 주문 수가 늘었는지 평균 주문 금액이 늘었는지, 특정 카테고리 매출이 늘었는지 추가로 확인해야 합니다.


## 14. 고객별 구매 금액 EDA

고객별 구매 금액은 우수 고객 분석의 출발점입니다. 하지만 한 번의 고액 구매 고객과 여러 번 반복 구매한 고객은 다르게 해석해야 합니다. 그래서 `total_sales`, `order_count`, `avg_order_value`를 함께 봅니다.


In [21]:
customer_sales_base = completed_items.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='many_to_one',
    indicator='_customer_merge',
)
customer_sales_base['city'] = customer_sales_base['city'].fillna('Unknown')

customer_sales = (
    customer_sales_base
    .groupby(['customer_id', 'city'], as_index=False, dropna=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)
customer_sales['avg_order_value'] = (
    customer_sales['total_sales'] / customer_sales['order_count']
).round(0)

print('고객 미매칭:', customer_sales_base['_customer_merge'].eq('left_only').sum())
customer_sales.head(10)

고객 미매칭: 1


,customer_id,city,order_count,total_sales,avg_order_value
1,3.0,서울,3,6702000.0,2234000.0
7,14.0,대구,4,5204000.0,1301000.0
57,90.0,수원,3,4311000.0,1437000.0
59,93.0,울산,4,4001000.0,1000250.0
47,65.0,광주,6,3977000.0,662833.0
66,104.0,서울,2,3966000.0,1983000.0
11,20.0,성남,4,3808000.0,952000.0
27,38.0,성남,4,3656000.0,914000.0
19,28.0,인천,4,3201000.0,800250.0
13,22.0,부산,3,3109000.0,1036333.0


In [22]:
source_total = float(completed_items['line_total'].sum())
total_validation = pd.DataFrame({
    'aggregation': ['source', 'category', 'monthly', 'customer'],
    'total_sales': [
        source_total,
        float(category_sales['total_sales'].sum()),
        float(monthly_sales['total_sales'].sum()),
        float(customer_sales['total_sales'].sum()),
    ],
})
total_validation['matches_source'] = total_validation['total_sales'].eq(source_total)
total_validation

,aggregation,total_sales,matches_source
0,source,141340000.0,True
1,category,141340000.0,True
2,monthly,141340000.0,True
3,customer,141340000.0,True


## 15. EDA 결과를 다음 질문으로 연결하기

EDA의 가치는 표를 만드는 데서 끝나지 않습니다. 각 결과가 다음 질문으로 이어져야 합니다.


In [23]:
eda_result_summary = pd.DataFrame({
    'question': [
        '고객은 어느 도시에 많이 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'result_table': [
        'customer_city',
        'product_category',
        'category_sales',
        'monthly_sales',
        'customer_sales',
    ],
    'next_question': [
        '도시별 구매 금액도 차이가 있는가?',
        '상품 수가 많은 카테고리가 매출도 높은가?',
        '매출 차이가 수량 때문인가 단가 때문인가?',
        '특정 월의 매출 변화는 어떤 카테고리 때문인가?',
        '고액 구매 고객은 반복 구매 고객인가?',
    ],
})

eda_result_summary


,question,result_table,next_question
0,고객은 어느 도시에 많이 분포하는가?,customer_city,도시별 구매 금액도 차이가 있는가?
1,어떤 카테고리의 상품이 많은가?,product_category,상품 수가 많은 카테고리가 매출도 높은가?
2,카테고리별 매출은 어떻게 다른가?,category_sales,매출 차이가 수량 때문인가 단가 때문인가?
3,월별 매출과 주문 수는 어떻게 변하는가?,monthly_sales,특정 월의 매출 변화는 어떤 카테고리 때문인가?
4,구매 금액이 높은 고객은 누구인가?,customer_sales,고액 구매 고객은 반복 구매 고객인가?


## 16. EDA 결과 저장하기

EDA 결과표를 CSV로 저장하면 다음 장의 시각화나 보고서 작성에서 다시 사용할 수 있습니다.


In [24]:
customer_city.to_csv(REPORT_DIR / 'ch06_customer_city.csv', index=False, encoding='utf-8-sig')
product_category.to_csv(REPORT_DIR / 'ch06_product_category.csv', index=False, encoding='utf-8-sig')
category_sales.to_csv(REPORT_DIR / 'ch06_category_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch06_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch06_customer_sales.csv', index=False, encoding='utf-8-sig')
eda_result_summary.to_csv(REPORT_DIR / 'ch06_eda_questions.csv', index=False, encoding='utf-8-sig')

list(REPORT_DIR.glob('ch06_*.csv'))


[PosixPath('/home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/reports/ch06_category_sales.csv'),
 PosixPath('/home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/reports/ch06_payment_sales.csv'),
 PosixPath('/home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/reports/ch06_product_category.csv'),
 PosixPath('/home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/reports/ch06_customer_city.csv'),
 PosixPath('/home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/reports/ch06_eda_questions.csv'),
 PosixPath('/home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/reports/ch06_monthly_sales.csv'),
 PosixPath('/home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/reports/ch06_customer_sales.csv')]

## 17. EDA 요약 보고서 만들기

간단한 Markdown 요약 보고서를 만들어 봅니다. 보고서에는 분석 목적, 주요 질문, 핵심 결과, 추가 질문, 해석 시 주의사항을 담습니다.


In [25]:
summary_text = f'''# Chapter 6 EDA 요약 보고서

## 분석 범위
- order_status == completed
- total_sales = completed 주문의 line_total 합계

## 질문
{questions.to_string(index=False)}

## 병합 및 총합 검증
{total_validation.to_string(index=False)}

## 카테고리별 결과
{category_sales.to_string(index=False)}

## 월별 결과
{monthly_sales.to_string(index=False)}

## 고객별 상위 결과
{customer_sales.head(10).to_string(index=False)}

## 한계
미매칭 키, 날짜 변환 실패, 분리된 카테고리 표기를 추가 확인해야 하며 total_sales를 회계상 순매출로 해석하지 않습니다.
'''

report_path = REPORT_DIR / 'ch06_eda_summary.md'
report_path.write_text(summary_text, encoding='utf-8')
print('EDA 요약 보고서 저장 완료:', report_path)

EDA 요약 보고서 저장 완료: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/reports/ch06_eda_summary.md


## 18. 소스 모듈로 같은 EDA 실행하기

위에서 노트북으로 한 단계씩 실행한 EDA는 `src/eda.py`에 함수로 정리되어 있습니다. 반복 실행하거나 프로젝트 코드로 관리하려면 노트북보다 소스 모듈을 사용하는 것이 좋습니다.


In [26]:
from src.eda import (
    build_eda_report,
    load_processed_sales_data,
    run_basic_eda,
    save_eda_outputs,
)

module_data = load_processed_sales_data(PROCESSED_DIR)
module_results = run_basic_eda(module_data)

module_results.keys()


dict_keys(['questions', 'customer_city', 'customer_gender', 'product_category', 'category_price', 'order_status', 'payment_method', 'completed_items', 'merge_checks', 'category_sales', 'monthly_sales', 'customer_sales', 'payment_sales', 'total_validation', 'eda_result_summary'])

In [27]:
module_results['category_sales'].head()


,category,total_quantity,total_sales,sales_ratio
10,패션,290.0,32940000.0,23.31
7,식품,226.0,25035000.0,17.71
9,전자기기,176.0,21485000.0,15.20
1,도서,174.0,17663000.0,12.50
3,뷰티,168.0,17103000.0,12.10


## 19. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 EDA 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/run_eda.py
```

이 스크립트는 6장 EDA 결과 CSV와 `reports/ch06_eda_summary.md`를 자동으로 저장합니다.


In [28]:
import subprocess

script_result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'run_eda.py')],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(script_result.stdout)

expected_outputs = [
    REPORT_DIR / name for name in [
        'ch06_category_sales.csv', 'ch06_monthly_sales.csv',
        'ch06_customer_sales.csv', 'ch06_eda_questions.csv',
        'ch06_customer_city.csv', 'ch06_product_category.csv',
        'ch06_payment_sales.csv', 'ch06_eda_summary.md',
    ]
]
pd.DataFrame({
    'file': [path.name for path in expected_outputs],
    'exists': [path.exists() for path in expected_outputs],
})

6장 EDA 완료

[completed 주문 Gate]
completed 상세 행: 446
completed 고유 주문 수: 179
completed 주문 기준 금액: 141340000

[병합 검증]
          check  rows_before  rows_after  unmatched
   orders merge          752         752          2
 products merge          446         446         16
customers merge          446         446          1

[카테고리별 매출 상위 5개]
category  total_quantity  total_sales  sales_ratio
      패션           290.0   32940000.0        23.31
      식품           226.0   25035000.0        17.71
    전자기기           176.0   21485000.0        15.20
      도서           174.0   17663000.0        12.50
      뷰티           168.0   17103000.0        12.10

[월별 매출]
order_month  total_sales  order_count  avg_order_value
    2025-07    9561000.0           11         869182.0
    2025-08    9362000.0           17         550706.0
    2025-09    5299000.0            4        1324750.0
    2025-10   12629000.0           17         742882.0
    2025-11    9583000.0           16         598938.0
    2025-12    8

,file,exists
0,ch06_category_sales.csv,True
1,ch06_monthly_sales.csv,True
2,ch06_customer_sales.csv,True
3,ch06_eda_questions.csv,True
4,ch06_customer_city.csv,True
5,ch06_product_category.csv,True
6,ch06_payment_sales.csv,True
7,ch06_eda_summary.md,True


## 20. LLM과 함께 질문을 확장하기

LLM은 EDA 질문을 확장하고 결과 해석 문장을 다듬는 데 도움이 됩니다. 하지만 LLM이 제안한 질문이 실제 데이터로 답할 수 있는지는 반드시 사람이 확인해야 합니다.

```text
온라인 쇼핑몰 데이터로 EDA를 수행하려고 합니다.

데이터셋:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total

요청:
1. 현재 데이터로 검증 가능한 EDA 질문 10개를 만들어 주세요.
2. 각 질문에 필요한 데이터셋과 pandas 기능을 함께 정리해 주세요.
3. 데이터에 없는 내용은 추측하지 마세요.
4. 각 질문을 계산 가능한 지표로 바꿔 주세요.
```


## 21. LLM 결과 해석 요청 예시

EDA 결과 해석을 요청할 때는 원인 단정을 막는 조건을 넣는 것이 좋습니다.

```text
다음은 온라인 쇼핑몰 카테고리별 매출 요약 결과입니다.

category,total_quantity,total_sales,sales_ratio
전자기기,320,12500000,42.5
생활용품,510,7800000,26.5
패션,260,6200000,21.1
식품,430,2900000,9.9

이 결과를 보고서에 넣을 수 있도록 해석해 주세요.

조건:
- 데이터에 없는 원인을 단정하지 말 것
- 원인 설명은 가설로 표현할 것
- 추가로 확인해야 할 분석 질문을 제안할 것
- 관찰, 가설, 추가 분석을 구분할 것
```


## 22. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
3. 결제수단별 평균 주문 금액을 계산하세요.
4. 월별·카테고리별 매출 요약표를 만들어 보세요.
5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.
6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.


In [29]:
# 과제 1. 도시별 고객 수와 completed 주문 금액 비교
city_sales = (
    customer_sales_base.groupby('city', as_index=False, dropna=False)
    .agg(total_sales=('line_total', 'sum'), order_count=('order_id', 'nunique'))
)
city_compare = (
    customer_city[['city', 'customer_count']]
    .merge(city_sales, on='city', how='outer')
    .fillna({'customer_count': 0, 'total_sales': 0, 'order_count': 0})
    .sort_values('total_sales', ascending=False)
)
city_compare

,city,customer_count,total_sales,order_count
5,부산,23,24916000.0,32
6,서울,18,22211000.0,20
2,광주,21,19866000.0,28
10,인천,16,15723000.0,23
3,대구,21,14781000.0,18
8,수원,10,12220000.0,13
9,울산,17,11545000.0,17
7,성남,9,11288000.0,13
1,고양,8,7222000.0,10
0,Unknown,2,1196000.0,2


In [30]:
# 과제 2. 카테고리별 상품 수와 completed 주문 금액 비교
category_compare = (
    product_category[['category', 'product_count']]
    .merge(category_sales, on='category', how='outer')
    .fillna({'product_count': 0})
    .sort_values('total_sales', ascending=False)
)
category_compare

,category,product_count,total_quantity,total_sales,sales_ratio
10,패션,16.0,290.0,32940000.0,23.31
7,식품,16.0,226.0,25035000.0,17.71
9,전자기기,13.0,176.0,21485000.0,15.20
1,도서,13.0,174.0,17663000.0,12.50
3,뷰티,16.0,168.0,17103000.0,12.10
6,스포츠,10.0,165.0,13237000.0,9.37
5,생활용품,9.0,108.0,7001000.0,4.95
0,Unknown,0.0,34.0,4015000.0,2.84
4,생활 용품,1.0,17.0,2108000.0,1.49
2,뷰 티,1.0,12.0,648000.0,0.46


In [31]:
# 과제 3. 결제수단별 completed 주문의 평균 주문 금액
payment_order_level = (
    completed_items.groupby(['order_id', 'payment_method'], as_index=False, dropna=False)
    .agg(order_total=('line_total', 'sum'))
)
payment_sales = (
    payment_order_level.groupby('payment_method', as_index=False, dropna=False)
    .agg(order_count=('order_id', 'nunique'), total_sales=('order_total', 'sum'))
    .sort_values('total_sales', ascending=False)
)
payment_sales['avg_order_value'] = (
    payment_sales['total_sales'] / payment_sales['order_count']
).round(0)
payment_sales

,payment_method,order_count,total_sales,avg_order_value
0,bank_transfer,49,39461000.0,805327.0
1,card,40,37542000.0,938550.0
2,kakao_pay,50,36861000.0,737220.0
3,naver_pay,40,27476000.0,686900.0


In [32]:
# 과제 4. 월별·카테고리별 completed 주문 금액
month_category_sales = (
    completed_items
    .merge(products[['product_id', 'category']], on='product_id', how='left', validate='many_to_one')
    .assign(
        category=lambda df: df['category'].fillna('Unknown'),
        order_month=lambda df: (
            pd.to_datetime(df['order_date'], errors='coerce')
            .dt.to_period('M').astype('string').fillna('Unknown')
        ),
    )
    .groupby(['order_month', 'category'], as_index=False, dropna=False)
    .agg(total_sales=('line_total', 'sum'))
)
assert month_category_sales['total_sales'].sum() == source_total
month_category_sales.pivot(
    index='order_month', columns='category', values='total_sales'
).fillna(0)

category,Unknown,도서,뷰 티,뷰티,생활 용품,생활용품,스포츠,식품,전자 기기,전자기기,패션
order_month,,,,,,,,,,,
2025-07,82000.0,1040000.0,0.0,1133000.0,0.0,406000.0,295000.0,2123000.0,0.0,1319000.0,3163000.0
2025-08,470000.0,1144000.0,0.0,1050000.0,0.0,983000.0,417000.0,505000.0,0.0,2547000.0,2246000.0
2025-09,780000.0,380000.0,0.0,125000.0,0.0,735000.0,438000.0,528000.0,0.0,1157000.0,1156000.0
2025-10,82000.0,1570000.0,0.0,661000.0,0.0,1334000.0,1426000.0,3088000.0,0.0,1781000.0,2687000.0
2025-11,420000.0,583000.0,108000.0,1473000.0,0.0,528000.0,1271000.0,2221000.0,0.0,1035000.0,1944000.0
2025-12,468000.0,34000.0,0.0,884000.0,0.0,489000.0,791000.0,2048000.0,0.0,1408000.0,2459000.0
2026-01,0.0,74000.0,0.0,376000.0,0.0,804000.0,2359000.0,2465000.0,105000.0,485000.0,1467000.0
2026-02,0.0,1827000.0,0.0,275000.0,124000.0,522000.0,1009000.0,1193000.0,0.0,1562000.0,1177000.0
2026-03,140000.0,1075000.0,0.0,728000.0,0.0,233000.0,364000.0,2792000.0,0.0,1378000.0,537000.0


## 제출 답안 요약

- GitHub ID: `chocoemong1`
- 작성일: 2026-10-08
- 최종 URL: [chapter06.ipynb](https://github.com/chocoemong1/llm-data-analysis-study/blob/main/chapter06/chapter06.ipynb)

### 1. 내가 정한 EDA 질문

| 질문 | 범위 | 지표 | 검증 |
| --- | --- | --- | --- |
| 카테고리별 금액·수량 차이는? | completed | total_sales, total_quantity | source 합계 |
| 월별 금액 변화는 주문 수와 객단가 중 무엇과 함께 변하는가? | completed | total_sales, order_count, avg_order_value | source 합계 |
| 결제수단별 주문 수와 객단가는? | completed | order_count, total_sales, avg_order_value | source 합계 |

![질문과 지표](images/step01_question_metric.png)

세 질문은 모두 현재 컬럼으로 계산할 수 있고, 금액 규모·주문 빈도·주문당 금액을 구분해 볼 수 있어 선택했다.

### 2. 기본 분포

![기본 분포](images/step02_basic_distribution.png)

- **관찰:** 부산 고객이 23명으로 가장 많고, 패션·뷰티·식품 상품은 각각 16개다. 주문은 completed 180건(60%)이다.
- **해석:** 고객 수·상품 수·주문 상태는 이후 금액 집계의 구성 배경이다.
- **한계:** 고객 또는 상품 수가 많다는 사실만으로 구매 금액이나 선호가 높다고 단정할 수 없다.

### 3. completed Gate와 병합 검증

![completed Gate](images/step03_completed_gate.png)

주문 상세 752행은 주문 병합 후에도 752행이며 미매칭은 2행이다. 그중 completed 범위는 446행·179개 주문·`line_total` 합계 141,340,000이다. 상품 미매칭 16행과 고객 미매칭 1행은 `Unknown`으로 보존해 총합 검증에서 빠지지 않게 했다.

### 4. 핵심 집계 결과

![핵심 집계](images/step04_group_analysis.png)

- 카테고리: 패션이 32,940,000(23.31%)·수량 290으로 가장 크다.
- 월: 2026-08이 16,980,000으로 가장 크며 주문 15건, 평균 주문 금액 1,132,000이다.
- 고객: 고객 ID 3은 3건·6,702,000·평균 2,234,000으로 총액이 가장 크다. 이름은 공개 결과에서 제외했다.

`total_sales`는 completed 주문의 `line_total` 합계이며 회계상 순매출이나 이익이 아니다.

### 5. 집계 총합 검증

![총합 검증](images/step05_validation.png)

| 기준 | 합계 |
| --- | ---: |
| completed 원본 | 141,340,000 |
| 카테고리 | 141,340,000 |
| 월 | 141,340,000 |
| 고객 | 141,340,000 |

네 합계가 모두 일치해 **PASS**로 판단했다. 결측 그룹을 제외했다면 합계가 달라졌을 것이므로 `Unknown`을 보존했다.

### 6. 관찰 → 가설 → 추가 검증

1. **관찰:** 패션은 금액과 수량이 모두 1위다. **가설:** 판매량과 단가가 함께 영향을 주었을 수 있다. **추가 검증:** 상품별 단가·수량·금액 집중도를 확인한다.
2. **관찰:** 2026-08은 주문 15건인데도 금액 1위다. **가설:** 주문당 금액이 큰 주문의 영향일 수 있다. **추가 검증:** 주문별 금액 분포와 카테고리 구성을 확인한다.
3. **관찰:** 고객 3은 3건으로 총액 1위지만 고객 65는 6건이어도 총액이 더 작다. **가설:** 구매 빈도보다 주문 크기의 영향이 클 수 있다. **추가 검증:** 주문별 금액과 구매 시점을 비교한다.

### 7. LLM 질문 확장과 나의 추가 EDA

- **Safe Context:** 데이터셋·컬럼, completed 범위, 익명화된 집계만 제공
- **Prompt:** 현재 데이터로 검증 가능한 추가 질문 3개와 지표·한계만 제안하고 원인을 단정하지 말 것
- **제안 요약:** 결제수단별 객단가, 도시별 고객당 금액, 카테고리 내 상품 집중도
- **판단:** 결제수단별 질문을 “completed 주문 기준”으로 수정해 사용했다. 필요한 컬럼이 있고 기존 카테고리·월 집계와 중복이 적기 때문이다.

![나의 추가 EDA](images/step06_my_question.png)

**결과 관찰:** bank_transfer의 총액은 39,461,000으로 가장 크지만 card의 평균 주문 금액은 938,550으로 가장 높다.

**해석:** 총액 1위와 객단가 1위는 다르므로 결제수단의 규모를 한 지표로 판단하면 안 된다.

**한계·다음 질문:** 수수료·고객 특성·결제 실패 정보가 없어 선호나 수익성을 단정할 수 없다. 결제수단별 주문 금액 분포와 이상치 민감도를 확인한다.

### 8. 최종 판단

가장 먼저 확인할 것은 ① 분리된 카테고리 표기(`전자기기/전자 기기` 등), ② 상품·고객 미매칭, ③ 날짜 실패와 고액 주문의 영향이다. 이는 EDA 중 값을 조용히 고치지 않고 Chapter 05 전처리 기준으로 되돌아가 검토해야 한다.

이번 실습에서 EDA는 표를 많이 만드는 작업이 아니라 **질문을 지표로 바꾸고, 같은 범위를 유지해 계산·검증한 뒤 다음 질문을 만드는 과정**이라고 판단했다.

#### 최종 체크

- [x] processed 데이터 4종 로딩
- [x] 질문 3개와 지표·범위·검증 연결
- [x] completed Gate와 안전한 병합
- [x] 카테고리·월·고객·추가 EDA 실행
- [x] 총합 검증 PASS
- [x] 관찰·가설·추가 검증 구분
- [x] 개인정보·Secret 제외

## 23. 정리

이번 장에서는 다음 내용을 실습했습니다.

- EDA에서 관찰, 가설, 결론 구분하기
- 현재 데이터로 답할 수 있는 질문 만들기
- 고객, 상품, 주문 데이터의 기본 분포 확인
- 주문 상세, 상품, 주문, 고객 데이터를 병합해 매출 분석하기
- 카테고리별 매출, 월별 매출, 고객별 구매 금액 계산하기
- EDA 결과를 다음 질문으로 연결하기
- EDA 결과 CSV와 Markdown 보고서 저장하기
- `src/eda.py`와 `scripts/run_eda.py`로 반복 실행 가능한 소스 구조 만들기

다음 장에서는 EDA에서 만든 질문과 집계표를 바탕으로 데이터 시각화를 다룹니다.
